<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP10.1B/MP10.1B_cell_ageing.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP10.1B · How Much Has the Cell Aged?

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# --- PyBaMM ------------------------------------------------------------------
# The single particle model with electrolyte is solved with PyBaMM (pybamm.org), which Colab does not have.
# This cell installs it once per session (about a minute); locally it does nothing if PyBaMM is there.
import importlib.util, subprocess, sys
if importlib.util.find_spec("pybamm") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybamm==26.10.1.1"], check=True)
import pybamm
print("PyBaMM", pybamm.__version__)

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.1-spm-and-thermal/")
PROJECT = (['mp101b.py', 'MP10.1B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP10.1B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex10.1-spm-and-thermal", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex10.1-spm-and-thermal", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp101b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp101b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP10.1B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex10.1's problem.py, by its name in Ex10.1

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

A lithium-ion cell loses capacity as it ages, and how it ages matters as much as how much. Two mechanisms dominate in the graphite electrode. **Lithium is lost**: it is bound in the film that grows on the graphite (the solid-electrolyte interphase), so less lithium can shuttle between the electrodes and the capacity falls at every rate. **Diffusion slows**: particles crack and films thicken, lithium moves more slowly through the graphite, and the cell loses capacity and voltage mostly at high rates. A battery management system, a second-life grading line or a warranty assessment needs to know both - the first says how much energy is left, the second how much power - and **can only measure the voltage of discharges**: a cell in service is not opened.

This is an **inverse problem**: Ex10.1 computed the concentration in a particle with a known diffusivity, and its section 6 found the diffusivity from readings of the surface concentration; here two hidden ageing factors are found from **terminal voltages**, which are what a cell tester records. And it is asked **again and again** - for every cell of a pack, at every check - so what matters is not only the answer but the time to it.

## A2 · The cell, and what is new against Ex10.1

The cell is the LG M50 21700 (NMC811 against graphite-SiOx, 5 Ah) in PyBaMM's `Chen2020` parameter set (Chen et al., J. Electrochem. Soc. 167, 2020): a different cell from Ex10.1's BYD 4680, chosen because its parameter set is complete and published for the model used here. The model is the **single particle model with electrolyte (SPMe)**: Ex10.1's particle for each electrode, a linearised electrolyte across the cell, and Butler-Volmer kinetics at the particles' surfaces; isothermal at 25 °C.

**What is new: two hidden ageing factors**, both of the graphite electrode:

| | value |
|---|---|
| the cell | LG M50, 5 Ah nominal (1C = 5 A), cut-off 2.5 V, 25 °C; graphite particles $R$ = 5.86 µm, $D_s$ = 3.3e-14 m²/s |
| $f_D$, slower diffusion | the graphite's diffusivity is $f_D\,D_s$; hidden, anywhere from 0.3 to 1.0 (assumed range) |
| $f_L$, lost lithium | the lithium in the graphite at full charge is $f_L$ times the fresh cell's; hidden, 0.80 to 1.00 (assumed range: up to 20 % of the cyclable lithium lost) |
| the records | constant-current discharges from full charge to 2.5 V at 0.5C, 1C or 2C: the voltage every 0.05 Ah discharged (1 % of the nominal capacity, every 36/C s), and the charge at the cut-off |
| the noise | 5 or 10 mV on every voltage, 5 mAh on the charge, one standard deviation, independent (assumed for a cell tester) |
| what is asked | $f_D$, $f_L$, and the capacity at 0.5C to 2.5 V |

The fresh cell is $f_D = f_L = 1$: that is the zero-strength check of part C.

In [ ]:
# The records of test case 1 against the fresh cell's
G = np.load("MP10.1B_truth.npz")
q = lambda V: G["dq"] * np.arange(1, V.size + 1)            # Ah, the charge at each reading
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for k, r in enumerate(G["rates"]):
    l, = ax[0].plot(q(G["fresh_V"][k]), G["fresh_V"][k], lw=1, label=f"fresh, {r:g}C")
    V = G["case1_V"][k]
    ax[0].plot(q(V), V, ".", ms=2, color=l.get_color(), label=f"case 1, {r:g}C")
    ax[1].plot(q(V), (V - G["fresh_V"][k]) * 1e3, ".", ms=2, color=l.get_color(), label=f"{r:g}C")
ax[0].set_xlabel("discharged charge q [Ah]"); ax[0].set_ylabel("terminal voltage V [V]"); ax[0].legend(fontsize=7, ncol=2)
ax[1].set_xlabel("q [Ah]"); ax[1].set_ylabel("case 1 - fresh cell [mV]"); ax[1].legend(); ax[1].set_ylim(-400, 50)
plt.tight_layout(); plt.show()
for k, r in enumerate(G["rates"]):
    print(f"{r:g}C: fresh cell {G['fresh_cap'][k]:.3f} Ah, case 1 {G['case1_cap'][k]:.3f} Ah to 2.5 V; "
          f"{np.isfinite(G['case1_V'][k]).sum()} voltage readings")

**What you should see.** Left, the fresh cell's three discharges and case 1's records (dots): the aged cell reaches 2.5 V earlier at every rate - 4.46 Ah instead of 5.02 at 0.5C. Right, the difference: within about 10 mV of the fresh cell through the first 2 Ah - the size of the noise, visible as the scatter of the dots - then growing to hundreds of millivolts where the aged cell's curve reaches its end first (down to 670 mV below the fresh cell at 0.5C). Most of what tells the ageing apart is in the last third of each discharge.

## A3 · The model and where the two factors enter

The SPMe is Ex10.1's particle twice - one representative particle per electrode - joined by the electrolyte and the reaction:

| what | equation | where the ageing enters |
|---|---|---|
| lithium in a graphite particle | $\dfrac{\partial c}{\partial t} = \dfrac{f_D D_s}{r^2}\dfrac{\partial}{\partial r}\Big(r^2\dfrac{\partial c}{\partial r}\Big)$, $\ -f_D D_s\dfrac{\partial c}{\partial r}\Big|_R = j_n$ | $f_D$: Ex10.1's equation with a smaller diffusivity |
| the start | $c = f_L\,c_{n,0}$ in the graphite; the NMC as fresh | $f_L$: less lithium to give |
| lithium in an NMC particle | the same equation with the NMC's data | - |
| the electrolyte | linearised about 1 000 mol/m³ across the cell | - |
| the voltage | $V = U_p(c_p|_R) - U_n(c_n|_R) + \eta_p - \eta_n + \Delta\phi_e - I R$ | through the surface concentrations |

with $j_n$ the flux of lithium out of the graphite's surface, set by the current, $U$ the open-circuit potentials and $\eta$ the Butler-Volmer overpotentials. The discharge stops at $V = 2.5$ V.

**Why there is no exact solution.** Ex10.1's particle at a constant flux has an exact series. Here the flux comes from Butler-Volmer kinetics and the voltage from nonlinear open-circuit potentials of both electrodes, coupled through the electrolyte: no series or closed form exists. The ground truth is numerical (part C).

**Ex10.1's series still says what $f_D$ does.** Once the flux is steady, the surface of a particle runs ahead of its mean by $jR/(5D)$: divide the diffusivity by two and the surface empties twice as far ahead. That surface is what the voltage sees.

In [ ]:
# How far the graphite's surface runs below its mean after an hour at 1C, from Ex10.1's exact series
for f in (1.0, 0.75, 0.5, 0.3):
    print(f"f_D = {f:.2f}: the surface is {mp.surface_lag(f):5.0f} mol/m³ below the mean "
          f"(of {mp.constants()['cmax_n']:.0f} mol/m³ the graphite can hold)")

**What you should see.** A fresh particle's surface sits 548 mol/m³ below its mean after an hour at 1C (1.7 % of the 33 133 mol/m³ the graphite can hold); at $f_D$ = 0.75 730, at 0.5 twice the fresh value (1 096), at 0.3 more than three times (1 826). Near the end of the discharge, where the graphite's open-circuit potential rises steeply as it empties, a few per cent of stoichiometry become tens to hundreds of millivolts.

## A4 · Why the inverse problem is hard

* **The two effects look alike.** Lost lithium shifts the graphite's stoichiometry window: its voltage change grows towards the end of the discharge, where the graphite's open-circuit potential rises steeply. Slower diffusion empties the surface ahead of the mean: also largest near the end. The cell below measures how alike the two voltage changes are.
* **One rate is not enough.** Lost lithium changes the curve at every rate in the same way; slower diffusion changes it more at a higher rate. Separating them needs at least two rates - case 4 of the test cases has one.
* **The noise.** 5 to 10 mV on every reading, against changes of a few tens of millivolts in most of the discharge.

In [ ]:
# How alike the two effects look: the voltage change of f_D = 0.5 alone against that of f_L = 0.9 alone
for r in G["rates"]:
    c, a, b = mp.shape_correlation(r)
    print(f"{r:g}C: slower diffusion alone changes the voltage by up to {a*1e3:4.0f} mV, lost lithium alone by up to "
          f"{b*1e3:4.0f} mV; the two changes correlate at {c:.2f}")

**What you should see.** At 0.5C slower diffusion moves the voltage by up to 26 mV and lost lithium by up to 462 mV; at 2C 96 and 343 mV. The shapes of the two changes correlate at 0.90, 0.91 and 0.92: from one discharge they are nearly the same signal, scaled. What separates them is how the diffusion effect grows with the rate while the lithium effect does not - and the capacity, which lost lithium sets at every rate.

## A5 · What is asked

Four test cases of rising difficulty, all public in `MP10.1B_truth.npz` - the records to fit, and the hidden answers to score against afterwards:

| case | rates recorded | noise | hidden: $f_D$, $f_L$ |
|---|---|---|---|
| 1 | 0.5C, 1C, 2C | 5 mV | in the file |
| 2 | 1C, 2C | 5 mV | in the file |
| 3 | 0.5C, 1C, 2C | 10 mV | in the file |
| 4 | 2C only | 10 mV | in the file |

| quantity | cases 1, 2 and 3 | case 4 (the 2C record only) |
|---|---|---|
| $f_D$ | within **5 %** | reported, not required |
| $f_L$ | within **1 %** | within **5 %** |
| the capacity at 0.5C to 2.5 V | within **1 %** | within **5 %** |
| the time to answer a new set of records | at least **10 times shorter** than a new least-squares fit with the reference model (part C) | the same |

One network, trained once over the whole range of ageing, answers a new set of records by being evaluated: **amortised** inference. Its training is reported apart, as the offline cost.

**The agreed accuracy, and why it differs from the first proposal.** The proposal asked for both factors within 5 % and the capacity within 1 % on case 1. That stands for $f_D$ and the capacity, and now for cases 2 and 3 too. $f_L$ is asked within 1 %, not 5 %: its whole range is 0.80 to 1.00, so 5 % would be half of it, and part C shows the records fix it to better than 0.1 %. Case 4 is new and harder: part C's Cramér-Rao bound shows that a single 2C record leaves $f_D$ with a standard deviation of 21 % and the capacity at 0.5C with 2.1 %, **whatever the method** - one rate cannot tell slower diffusion from lost lithium. There $f_L$ and the capacity are asked within 5 % (about two standard deviations of the bound) and $f_D$ is reported only. The least-squares reference and the solved network meet these on all four cases.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

An amortised network inverts the records directly:

$$\text{voltages at } 64 \text{ readings} \times 3 \text{ rates},\ \text{the charge at each cut-off},\ \text{which rates were recorded}\ \longmapsto\ (f_D,\ f_L,\ \text{capacity at 0.5C})$$

The first 64 readings (to 3.2 Ah) are present in every record over the range of ageing - the most aged cell reaches 2.5 V after 3.33 Ah at 2C - so every input has a fixed place. The readings beyond them, and the charge at the cut-off, carry the end of the discharge; the charge is an input of its own.

Trained on many cells whose ageing is known, the network learns the map from records to factors once. A new set of records then costs one evaluation, against the tens of forward solves a least-squares fit needs.

Its physics comes from the **forward model**: every training example is a cell simulated by the SPMe of part C at random factors. Noise is added afresh at every step of training, and a random subset of the three rates is removed, so the network learns to read through the noise and from any rates recorded. A **physics-informed** alternative - an inverse PINN of the SPMe with the two factors trainable, fitted to one set of records - answers one set at a time and must be trained again for every new set; you may compare it in question 9.

The scales of the problem:

In [ ]:
# The scales: the fresh cell's records, and how much the corners of the range change them
ref = mp.reference_model()                                # the forward model on the default mesh (part C)
for fD, fL in ((1.0, 1.0), (0.3, 1.0), (1.0, 0.8), (0.3, 0.8)):
    caps = [ref.record(fD, fL, r)[1] for r in mp.RATES]
    d2 = ref.record(fD, fL, 2.0)[0][:mp.N_READ] - ref.record(1.0, 1.0, 2.0)[0][:mp.N_READ]
    print(f"f_D {fD:.1f}, f_L {fL:.1f}: capacity at 0.5 / 1 / 2C {caps[0]:.2f} / {caps[1]:.2f} / {caps[2]:.2f} Ah; "
          f"largest change of the first 64 readings at 2C {np.abs(d2).max()*1e3:4.0f} mV")

**What you should see.** The fresh cell delivers 5.02, 4.94 and 4.76 Ah at 0.5, 1 and 2C. The slowest diffusion alone ($f_D$ = 0.3) costs 0.11 Ah at 0.5C but 0.42 Ah at 2C - a rate effect; the most lithium lost alone ($f_L$ = 0.8) costs about 1.0 Ah at every rate. Both together leave 3.33 Ah at 2C. Within the first 64 readings at 2C they change the voltage by up to 51, 143 and 452 mV, against a noise of 5 to 10 mV.

## B2 · The physics the training data obey

| what | how it enters |
|---|---|
| the SPMe with the two factors | each training example is one forward solve per rate, so every example satisfies it |
| the readings | the solver's own interpolation at the instants a tester would read, every 0.05 Ah |
| the noise | added at every training step: a voltage noise drawn from 5 to 10 mV per example, 5 mAh on the charge |
| missing rates | each example keeps a random non-empty subset of the three rates; the missing ones are zeroed and flagged |
| the capacity at 0.5C | a target beside the factors: it is what the operator needs, and the 0.5C record holds it when that rate was recorded |

## B3 · The training data

`mp.sample_factors(n, seed)` draws cells uniformly over the ranges; `mp.simulate(factors)` solves each one at the three rates on the reference mesh and returns its 64 first readings at each rate (n, 3, 64) and its charge at each cut-off (n, 3). About 10 ms per discharge: 16 000 cells took 8 to 12 minutes in our runs, a cost paid once, offline, like the training. The cell below simulates a few to show them; part D simulates the training set.

In [ ]:
# A few training examples: cells drawn over the range, and their records
F = mp.sample_factors(300, seed=1)
t0 = time.perf_counter()
Rd, Cd = mp.simulate(F, verbose=False)
print(f"{len(F)} cells in {time.perf_counter() - t0:.1f} s")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].scatter(F[:, 0], F[:, 1], c=Cd[:, 0], s=8, cmap="viridis"); ax[0].set_xlabel("f_D"); ax[0].set_ylabel("f_L")
ax[0].set_title("cells, coloured by the capacity at 0.5C", fontsize=9)
ax[1].scatter(F[:, 1], Cd[:, 0], s=4, label="0.5C"); ax[1].scatter(F[:, 1], Cd[:, 2], s=4, label="2C")
ax[1].set_xlabel("f_L"); ax[1].set_ylabel("charge at the cut-off [Ah]"); ax[1].legend()
ax[2].scatter(F[:, 0], Cd[:, 0] - Cd[:, 2], s=4); ax[2].set_xlabel("f_D"); ax[2].set_ylabel("capacity at 0.5C - at 2C [Ah]")
plt.tight_layout(); plt.show()

**What you should see.** 300 cells in about 7 to 12 s. Left, the cells spread over the two factors. Middle, the charge at the cut-off falls almost in proportion to $f_L$ at both rates: lost lithium is capacity lost. Right, the gap between the 0.5C and the 2C capacity grows as $f_D$ falls, from about 0.24 Ah for a nearly fresh particle to 0.53 Ah at $f_D$ near 0.3: slower diffusion is capacity lost at high rate. These two plots are why the factors can be told apart from several rates and hardly from one.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Three solvers, three jobs

| | the **forward model** | the **reference model** | the **ground truth** |
|---|---|---|---|
| what it is | PyBaMM's SPMe, built once with $f_D$, $f_L$ and the current as inputs | least squares over the two factors, the forward model on the default mesh inside | the forward model on a fine mesh |
| mesh | any | 20 points in each particle's radius and in each layer | 160 in each, tolerances 10⁻⁹ |
| its job | to make the training data | to be **timed** against the network: the online cost of a new set of records | to make the test cases' records and to **score** the answers |

## C2 · The forward model, step by step

1. **Two particles.** Each electrode is one spherical particle, cut into shells (finite volumes, Ex10.1's solver); the flux through the outer shell is the reaction, set by the current.
2. **The electrolyte.** Its concentration across the cell is the linear response to the current (the SPMe's correction to the single particle model), on 20 cells in each layer.
3. **The voltage.** The open-circuit potentials at the particles' surfaces, the Butler-Volmer overpotentials, the electrolyte's potential drop and the contact resistance.
4. **Time.** SUNDIALS' IDA solver adapts its step; the readings are its own interpolation at the instants a tester reads (every 0.05 Ah), not a straight line drawn afterwards between its steps - which on a steep part of the curve would be off by tens of millivolts.
5. **The cut-off** is an event: the solver stops when the voltage reaches 2.5 V, and the charge passed to then is the capacity.
6. **Built once.** $f_D$, $f_L$ and the current are inputs of the built model, so a new cell costs one solve per rate, about 10 ms.

## C3 · The reference model: least squares

`mp.least_squares_fit(records, noise)` finds the factors whose predicted records are closest to the measured ones, each voltage residual divided by the noise and each charge by 5 mAh, with `scipy.optimize.least_squares` (trust-region reflective within bounds, derivatives by finite differences), over $(\ln f_D, f_L)$ - the logarithm because $f_D$ acts through $1/f_D$ over a range of three. It starts from the middle of the range.

In [ ]:
# The ground truth: convergence on case 1's cell at 2C, the balances and the zero-strength check, from the truth file
print("mesh   points   capacity [Ah]   V at 1 Ah [V]   V at 3 Ah [V]   solve [s]")
for r, n, c, v1, v3, s in zip(G["conv_refine"], G["conv_points"], G["conv_cap"], G["conv_V1"], G["conv_V3"], G["conv_seconds"]):
    print(f" x{int(r)}     {int(n):4d}      {c:.5f}        {v1:.5f}         {v3:.5f}       {s:.2f}")
print("capacity change between meshes [Ah]:", " ".join(f"{x:+.6f}" for x in np.diff(G["conv_cap"])),
      "| observed order", np.round(G["conv_order"], 2), f"| Richardson {float(G['conv_richardson_cap']):.5f} Ah")
print(f"lithium out of the graphite against the charge passed (Faraday): {float(G['balance_faraday']):.1e}; "
      f"lithium in the particles kept to {float(G['balance_particles']):.1e}")
print(f"f_D = f_L = 1 against the fresh Chen2020 cell: {float(G['zero_strength_error_V']):.1e} V at most")

**What you should see.** Four meshes, each twice as fine: the capacity changes by 0.40, 0.10 and 0.025 mAh, four times less each time - **second order** (observed 1.99 and 2.00); the ground truth (x8) is within 0.01 mAh of the Richardson value, and its voltage at 1 and 3 Ah within 0.03 mV of the x4 mesh's: two hundred times finer than the 5 mV noise. The lithium that leaves the graphite equals the charge passed over Faraday's constant to the solver's tolerance, and the lithium in the particles is kept. At $f_D = f_L = 1$ the model with the ageing inputs gives the fresh cell's discharges to $10^{-6}$ V: the zero-strength check.

The reference model's own mesh is coarser: at the same factors its readings differ from the ground truth's by up to 1.3, 2.7 and 4.1 mV at 0.5, 1 and 2C, smaller than the noise.

In [ ]:
# The four test cases: least squares with the reference model on each, and its time
CASES = [str(c) for c in G["cases"]]

def records(n):
    '''Case n's records as (rate, voltages, charge at the cut-off), for the rates recorded.'''
    out = []
    for k, r in enumerate(G["rates"]):
        if G[f"case{n}_mask"][k]:
            V = G[f"case{n}_V"][k]
            out.append((float(r), V[np.isfinite(V)], float(G[f"case{n}_cap"][k])))
    return out

LSQ = {}
print("case rates  noise | f_D error   f_L error   capacity error | forward solves  time [s]")
for n in CASES:
    f = mp.least_squares_fit(records(n), float(G[f"case{n}_noise"]))
    e = (f["f_D"] / float(G[f"case{n}_f_D"]) - 1, f["f_L"] / float(G[f"case{n}_f_L"]) - 1, f["cap"] / float(G[f"case{n}_cap05"]) - 1)
    LSQ[n] = dict(f, err=e)
    print(f"  {n}    {int(G[f'case{n}_mask'].sum())}    {float(G[f'case{n}_noise'])*1e3:3.0f} mV | {e[0]*100:+6.2f} %   {e[1]*100:+6.2f} %     "
          f"{e[2]*100:+6.2f} %       |      {f['solves']:4d}       {f['seconds']:5.2f}")
t_lsq = float(np.median([v["seconds"] for v in LSQ.values()]))
REF = {"t_ref_s": t_lsq, "lsq": LSQ}                     # kept for part E
print(f"\nleast squares: {t_lsq:.2f} s per new set of records (the median of the four); "
      f"the network must answer in under {t_lsq*1e3/10:.0f} ms")

**What you should see.** Least squares finds $f_D$ within 0.1 to 1.4 %, $f_L$ within 0.16 % and the capacity at 0.5C within 0.12 % on all four cases - even case 4's $f_D$ (+1.3 %), which the bound below shows is luck of the noise drawn, not something a single 2C record can promise. It needs 46 to 82 forward solves (each a discharge at one rate on the default mesh), 0.5 to 1.1 s per set of records on one CPU core (median 0.64 and 0.72 s in two runs); this runtime's median is kept in `REF`. The requirements of part A were set from these numbers and the bound below.

In [ ]:
# What the records can tell: the Cramér-Rao bound at each case's true factors
print("case | smallest possible standard deviation of:  f_D [%]   f_L [%]   capacity at 0.5C [%] | correlation of the two")
for n in CASES:
    rates = [float(r) for k, r in enumerate(G["rates"]) if G[f"case{n}_mask"][k]]
    b = mp.information_bound(float(G[f"case{n}_f_D"]), float(G[f"case{n}_f_L"]), rates, float(G[f"case{n}_noise"]))
    print(f"  {n}  |                                          {b[0]*100:6.2f}    {b[1]*100:6.3f}       {b[2]*100:6.3f}          |     {b[3]:+.4f}")

**What you should see.** The **Cramér-Rao bound**: from how fast the records change as each factor changes (the Jacobian $J$, by central differences) and the noise $\sigma$, the covariance $\sigma^2 (J^\top J)^{-1}$ is the smallest any unbiased method can reach - a statement about the records, not about a method. With two or three rates: $f_D$ to 0.59 to 1.46 %, $f_L$ to 0.03 to 0.06 %, the capacity to 0.02 to 0.04 %, the two factors' estimates correlating at -0.84 to -0.94. With the 2C record alone (case 4): $f_D$ to 21 %, $f_L$ to 2.6 %, the capacity at 0.5C to 2.1 %, and a correlation of -0.9998 - one rate fixes one combination of the two factors, not both. The requirements of part A were set from these numbers.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2, including how many cells you simulate), and the **figures** (TODO 3).

**The contract.** Your network is a PyTorch module called `net`, used as `net(early, late, charges, mask)`:

| argument | meaning | shape |
|---|---|---|
| `early` | the first 64 voltage readings [V] of each rate (0.5C, 1C, 2C), every 0.05 Ah from 0.05 Ah; zero where a rate was not recorded | `(n, 3, 64)` |
| `late` | the last 24 readings [V] before each cut-off: the end of the discharge, where the two factors differ most; zero where not recorded | `(n, 3, 24)` |
| `charges` | the charge at the 2.5 V cut-off [Ah] of each rate; zero where not recorded | `(n, 3)` |
| `mask` | 1 for a rate that was recorded, 0 otherwise | `(n, 3)` |
| returns | $f_D$, $f_L$, the capacity at 0.5C [Ah] | `(n, 3)` |

Whatever you compute inside - features of the records, scaling, several heads - stays inside `net`.

## D2 · The training data and the loss, given

`make_data(n, seed)` simulates n cells with the forward model (part B) and returns the clean records, charges and targets as tensors; `noisy(records, charges, generator)` adds fresh noise, cuts the two windows (`windows`) and removes a random subset of the rates. `loss(net, early, late, charges, mask, targets)` is the mean squared error of the three outputs, $f_D$ compared through its logarithm (its range is a factor of three), each scaled by its range.

In [ ]:
# The training data and the loss, given ---------------------------------------------
def windows(records):
    """The two windows of records (n, 3, 110) [V], NaN past each cut-off and for a rate not recorded:
    the first 64 readings and the last 24 before the cut-off, each (n, 3, ·), zero where not recorded."""
    n = torch.isfinite(records).sum(-1, keepdim=True)                    # readings per record
    idx = (n - mp.N_LATE + torch.arange(mp.N_LATE, device=records.device)).clamp(min=0)
    late = torch.gather(records, -1, idx)
    return torch.nan_to_num(records[..., :mp.N_READ]), torch.nan_to_num(late)

_fresh = [mp.reference_model().record(1.0, 1.0, r) for r in mp.RATES]
_FR = np.full((1, 3, mp.N_MAX), np.nan)
for k, (v, _) in enumerate(_fresh):
    _FR[0, k, :len(v)] = v
FRESH_E, FRESH_L = (w[0] for w in windows(torch.tensor(_FR, device=DEVICE)))  # (3, 64), (3, 24) fresh cell's windows [V]
FRESH_C = torch.tensor([c for _, c in _fresh], device=DEVICE)                  # (3,) fresh cell's charges [Ah]
SUBSETS = torch.tensor([[(m >> k) & 1 for k in range(3)] for m in range(1, 8)], dtype=torch.float64, device=DEVICE)
# how often each subset of rates is kept: one rate 5 % each, two rates 15 % each, all three 40 % (a single rate
# cannot separate the two factors, part C, so it gets less of the training)
P_SUBSET = torch.tensor([0.05 if int(m.sum()) == 1 else 0.15 if int(m.sum()) == 2 else 0.40 for m in SUBSETS], device=DEVICE)

def make_data(n, seed):
    """n cells with random ageing, simulated: records (n, 3, 110) [V], charges (n, 3) [Ah], targets (n, 3)."""
    F = mp.sample_factors(n, seed)
    R_, C_ = mp.simulate(F, verbose=True)
    targets = np.c_[F, C_[:, 0]]                         # f_D, f_L, the capacity at 0.5C (the first rate's charge)
    return (torch.tensor(R_, device=DEVICE), torch.tensor(C_, device=DEVICE), torch.tensor(targets, device=DEVICE))

def noisy(records, charges, sigma=(0.005, 0.010), generator=None):
    """The records with fresh noise (a voltage σ drawn per example, 5 mAh on the charges) and a random non-empty
    subset of the rates kept (P_SUBSET; the others zeroed and flagged). Returns early, late, charges, mask."""
    n = records.shape[0]
    s = sigma[0] + (sigma[1] - sigma[0]) * torch.rand(n, 1, 1, generator=generator, device=DEVICE)
    early, late = windows(records + s * torch.randn(records.shape, generator=generator, device=DEVICE))
    c = charges + mp.CAP_NOISE * torch.randn(charges.shape, generator=generator, device=DEVICE)
    mask = SUBSETS[torch.multinomial(P_SUBSET, n, replacement=True, generator=generator)]
    return early * mask[:, :, None], late * mask[:, :, None], c * mask, mask

TSCALE = torch.tensor([0.6, 0.1, 0.5], device=DEVICE)   # the ranges of ln f_D, f_L and the capacity [Ah]

def loss(net, early, late, charges, mask, targets):
    """Mean squared error of ln f_D, f_L and the capacity, each scaled by its range."""
    p = net(early, late, charges, mask)
    e = torch.stack([torch.log(p[:, 0]) - torch.log(targets[:, 0]), p[:, 1] - targets[:, 1], p[:, 2] - targets[:, 2]], 1)
    return ((e / TSCALE) ** 2).mean()

## D3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The readings are 2.5 to 4.2 V, but what the ageing does is a change of millivolts to a few hundred millivolts on top: the readings relative to the fresh cell's (`FRESH_E`, `FRESH_L`, `FRESH_C`) are the signal. A missing rate must be told apart from a reading of zero (the mask);
* **what it returns, and how.** Scaled outputs; $f_D$ must stay positive;
* **its size.** Layers, neurons, activation.

Remember part A, section 4: the two factors are told apart by how the change grows with the rate, and by the charges.

In [ ]:
# TODO 1 - your network. net(early (n, 3, 64), late (n, 3, 24) [V], charges (n, 3) [Ah], mask (n, 3)) -> (n, 3): f_D, f_L, capacity [Ah]
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def forward(self, early, late, charges, mask):
        ...                                              # return (n, 3)

torch.manual_seed(101)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Simulate your training set with `make_data` (decide how many cells: about 30 ms each on a CPU for the three rates, and the simulation is part of the offline cost), then train `net` with `noisy` and `loss`. Decide the optimiser, the epochs, the batch size, the learning rate and its schedule, and whether to keep cells apart for validation. Keep the loss history for TODO 3, and **measure the offline time**, simulation included: set `TRAIN_SECONDS`.

`quick_score()` below gives, over the four test cases, the network's errors of $f_D$, $f_L$ and the capacity at 0.5C [%], one row per case.

In [ ]:
# A quick score, given: the errors on the four test cases [%]
CASE_E, CASE_L = windows(torch.tensor(np.stack([G[f"case{n}_V"] for n in CASES]), device=DEVICE))   # (4, 3, 64), (4, 3, 24) [V]
CASE_C = torch.tensor(np.stack([np.nan_to_num(G[f"case{n}_cap"]) for n in CASES]), device=DEVICE)    # (4, 3) [Ah]
CASE_M = torch.tensor(np.stack([G[f"case{n}_mask"] for n in CASES]), device=DEVICE)                 # (4, 3)

def answer(early=CASE_E, late=CASE_L, charges=CASE_C, mask=CASE_M):
    """The network's answer to sets of records: (n, 3) as a NumPy array."""
    with torch.no_grad():
        return net(early, late, charges, mask).cpu().numpy()

def case_errors(p):
    """Per case: the relative errors of f_D, f_L and the capacity at 0.5C [-]."""
    return np.array([(p[i, 0] / float(G[f"case{n}_f_D"]) - 1, p[i, 1] / float(G[f"case{n}_f_L"]) - 1,
                      p[i, 2] / float(G[f"case{n}_cap05"]) - 1) for i, n in enumerate(CASES)])

def quick_score():
    return np.round(case_errors(answer()) * 100, 2)

In [ ]:
# TODO 2 - your training
history = []
t0 = time.perf_counter()
...                                                      # simulate your training set, then train; append the loss to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"offline {TRAIN_SECONDS:.0f} s; errors [%] of f_D, f_L, capacity per case:"); print(quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss during training, and the network's answers against the truth on held-out cells ($f_D$, $f_L$, capacity), with all three rates and with one. Save each figure you want in the report as `MP10.1B_report_<name>.png` (in `cc.output_path`).

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores and keeps the answers in `R`: its answers to the four test cases, its answers to 100 fresh noisy sets of case 1's records (to see the spread the noise causes), and the **online time** for 1, 10 and 100 new sets of records, against a new least-squares fit (part C's time, measured on this runtime).

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
out = {"cases": np.array(CASES), "answer": answer()}                    # (4, 3)
# 100 new sets of records of case 1: the same cell, fresh noise of its σ
rng = np.random.default_rng(100)
clean = G["case1_clean"]                                 # (3, 110), all three rates
rep_V = clean[None] + float(G["case1_noise"]) * rng.standard_normal((100,) + clean.shape)
rep_C = G["case1_clean_cap"][None] + mp.CAP_NOISE * rng.standard_normal((100, 3))
RE, RL = windows(torch.tensor(rep_V, device=DEVICE))
RC, RM = torch.tensor(rep_C, device=DEVICE), torch.ones(100, 3, device=DEVICE)
out["case1_repeat"] = answer(RE, RL, RC, RM)
timing = {}
for k in (1, 10, 100):
    timing[f"net_{k}"] = middle(lambda: answer(RE[:k], RL[:k], RC[:k], RM[:k]))
out.update({k: np.array(v) for k, v in timing.items()})
out["lsq_seconds"] = np.array(REF["t_ref_s"])
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP10.1B_net.pt"))
print(f"online, one new set of records: network {timing['net_1']*1e3:.2f} ms, least squares {REF['t_ref_s']*1e3:.0f} ms "
      f"({REF['t_ref_s']/timing['net_1']:.0f} times)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the hidden values in the truth file `G` and the least-squares reference `REF` of part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP10.1B_truth.npz")
print("in R:", ", ".join(R))
print(f"least squares (part C): {REF['t_ref_s']:.2f} s per new set of records")

## E2 · TODO 4 · The goal metrics

Compute, for each of the four cases, and put in `METRICS` (a name and a number each):

* the errors of $f_D$, $f_L$ and the capacity at 0.5C [%], the network's and the least-squares reference's (`REF["lsq"]`);
* the network's online time for one set of records, the least-squares time, and their ratio;
* the offline time.

Then say in one line each whether the requirements of part A are met (`mp.TOL` holds them per case).

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>50}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP10.1B_report_<name>.png`, the figures your answers refer to. At least: each case's true factors, the network's and the least-squares' in the $(f_D, f_L)$ plane; the spread of the network's answers to the 100 noisy repeats of case 1 (`R["case1_repeat"]`) against the Cramér-Rao bound; and the online times for 1, 10 and 100 sets of records against least squares.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 5 are about your network and its result; 6 to 9 ask you to use it; 10 concludes. The next cell writes `MP10.1B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: its inputs (and any features computed from the records), layers, neurons, outputs and number of weights. Why this design?": """
""",
    "2. Your training data and training: how many cells you simulated and how long that took, the noise and missing rates, the optimiser, epochs and the offline time in seconds.": """
""",
    "3. The four test cases: give the network's errors of f_D, f_L and the capacity at 0.5C [%] for each. Which requirements are met?": """
""",
    "4. The same errors for the least-squares reference: on which cases is the network better, on which worse, and why?": """
""",
    "5. The online time for one new set of records: the network against a new least-squares fit, in milliseconds, and the ratio. Then for 10 and 100 sets: how do the two grow?": """
""",
    "6. Case 1 repeated 100 times with fresh noise: how large is the spread of the network's f_D, f_L and capacity, and its mean error? Compare them with the Cramér-Rao bound of part C.": """
""",
    "7. Case 4 has the 2C record only: what do the network and least squares give for f_D and f_L, and what does the bound's correlation of -0.9998 say about why?": """
""",
    "8. Over your held-out cells: how do the errors depend on the number of rates recorded and on f_D itself (fresh against strongly aged particles)? Which cells can your network not diagnose?": """
""",
    "9. Choose one case and fit it with an inverse physics-informed network (the SPMe's particle with f_D and f_L trainable), or argue with numbers why it would be slower: what is its online cost for one new set of records?": """
""",
    "10. To conclude: a grading line receives 2 000 second-life cells a day, each discharged at two rates. Which method would you give it, given your measured accuracy, online and offline times? What would change for a cell of another chemistry?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP10.1B — How Much Has the Cell Aged?", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP10.1B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP10.1B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP10.1B_report.md into MP10.1B_report.pdf, with any figure
# saved as MP10.1B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP10.1B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP10.1B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP10.1B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP10.1B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass